In [1]:
import glob
import json
import os
import numpy as np
import rasterio
from pathlib import Path

# --- Perbaikan 2026-11-14: cari repo root secara robust agar notebook tidak
# tergantung pada cwd saat dieksekusi (nbconvert memakai cwd notebook/,
# Jupyter/VS Code biasanya memakai cwd repo root). ---
def _find_repo_root():
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "datasets").is_dir() and (
            list((cand / "datasets").glob("*.tif"))
            or list((cand / "datasets").glob("*.TIF"))
        ):
            return cand
        if (cand / ".git").exists():
            return cand
    return start

REPO_ROOT = _find_repo_root()
DATASETS_DIR = REPO_ROOT / "datasets"

# 1. Otomatis cari file .tif di folder 'datasets/' terlebih dahulu, lalu fallback ke folder utama
# Cari .tif di datasets/ di bawah repo root (bukan relatif terhadap cwd).
tif_files = (
    glob.glob(str(DATASETS_DIR / "*.tif")) +
    glob.glob(str(DATASETS_DIR / "*.TIF")) +
    glob.glob(str(REPO_ROOT / "*.tif")) +
    glob.glob(str(REPO_ROOT / "*.TIF"))
)

if not tif_files:
  raise FileNotFoundError(
      "File .tif tidak ditemukan di folder utama maupun di folder 'datasets/'! "
      "Pastikan file GeoTIFF Anda berada di lokasi yang tepat."
  )

tif_path = tif_files[0]
print(f"[1/4] Membaca raster GeoTIFF: {tif_path} ...")

with rasterio.open(tif_path) as src:
  band = src.read(1)
  nodata = src.nodata
  height, width = band.shape
  bounds = src.bounds
  res_x, res_y = src.res
  print(f"      Dimensi Raster: {width}x{height} piksel | Resolusi: {res_x:.4f} derajat")

  def get_density(lat, lon):
    """Ambil nilai kepadatan penduduk (jiwa/km2) tepat pada koordinat (lat, lon)."""
    try:
      if lat is None or lon is None:
        return 0.0
      lat_f, lon_f = float(lat), float(lon)
      if np.isnan(lat_f) or np.isnan(lon_f):
        return 0.0
      r, c = src.index(lon_f, lat_f)
      r = max(0, min(height - 1, r))
      c = max(0, min(width - 1, c))
      val = float(band[r, c])
      if (nodata is not None and val == nodata) or val < 0 or np.isnan(val):
        return 0.0
      return round(val, 2)
    except Exception:
      return 0.0

  # 2. Ekstrak seluruh profil garis lintang (-90 s/d +90 per 1 derajat)
  print("[2/4] Mengekstrak profil kepadatan penduduk seluruh pita garis lintang (-90° s/d 90°)...")
  lat_profile_1deg = []
  for lat_deg in range(-90, 90):
    r1, _ = src.index(0, min(bounds.top - 1e-5, lat_deg + 1))
    r2, _ = src.index(0, max(bounds.bottom + 1e-5, lat_deg))
    r_start, r_end = min(r1, r2), max(r1, r2) + 1
    slice_band = band[r_start:r_end, :]

    valid_mask = (~np.isnan(slice_band)) & (slice_band >= 0)
    if nodata is not None:
      valid_mask = valid_mask & (slice_band != nodata)

    valid_vals = slice_band[valid_mask]
    lat_profile_1deg.append({
        "lat_deg": lat_deg,
        "mean_pop_density_km2": (
            round(float(np.mean(valid_vals)), 2) if len(valid_vals) > 0 else 0.0
        ),
        "max_pop_density_km2": (
            round(float(np.max(valid_vals)), 2) if len(valid_vals) > 0 else 0.0
        ),
        "inhabited_cells": int(np.sum(valid_vals > 0)),
    })

  # 3. Ekstrak Grid Global Berpenghuni (agar user bisa klik titik MANA SAJA di peta Mode 3)
  print("[3/4] Mengekstrak matriks grid global (0.5° x 0.5°) untuk fitur klik bebas di peta...")
  step_deg = 0.5
  global_grid_cells = []
  for lat_g in np.arange(-60.0, 75.0, step_deg):
    for lon_g in np.arange(-180.0, 180.0, step_deg):
      r1, c1 = src.index(lon_g, lat_g + step_deg)
      r2, c2 = src.index(lon_g + step_deg, lat_g)
      sub = band[min(r1, r2) : max(r1, r2) + 1, min(c1, c2) : max(c1, c2) + 1]
      mask = (~np.isnan(sub)) & (sub > 0.5)
      if nodata is not None:
        mask = mask & (sub != nodata)
      if np.any(mask):
        mean_val = round(float(np.mean(sub[mask])), 1)
        # Disimpan dalam format ringkas [lat, lon, pop_density_km2]
        global_grid_cells.append(
            [round(float(lat_g), 2), round(float(lon_g), 2), mean_val]
        )

  with open(DATASETS_DIR / "sedac_global_grid_lookup.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "grid_resolution_deg": step_deg,
            "format": "[lat, lon, pop_density_km2]",
            "total_inhabited_cells": len(global_grid_cells),
            "cells": global_grid_cells,
        },
        f,
    )
  print(f"      -> Tersimpan: sedac_global_grid_lookup.json ({len(global_grid_cells)} grid daratan berpenghuni)")

  # 4. Update seluruh data di meteorite_datamining_lovable.json
  lovable_json_path = REPO_ROOT / "meteorite_datamining_lovable.json"
  if os.path.exists(lovable_json_path):
    print(f"[4/4] Menggabungkan seluruh data SEDAC ke {lovable_json_path} ...")
    with open(lovable_json_path, "r", encoding="utf-8") as f:
      app_data = json.load(f)

    # a. Semua 19 meteorit Indonesia
    for item in app_data.get("indonesia_meteorites", []):
      item["pop_density_km2"] = get_density(item.get("lat"), item.get("lon"))

    # b. Semua 511 pin peta dunia
    for item in app_data.get("map_meteorites", []):
      item["pop_density_km2"] = get_density(item.get("lat"), item.get("lon"))

    # c. Semua centroid klaster K-Means
    for cl in app_data.get("kmeans_clusters", []):
      cl["centroid_pop_density_km2"] = get_density(
          cl.get("centroid_lat"), cl.get("centroid_lon")
      )

    # d. Daftar lengkap kota Indonesia & Dunia untuk Preset Mode 3
    all_target_cities = [
        {"name": "Jakarta (Monas)", "country": "Indonesia", "lat": -6.1754, "lon": 106.8272},
        {"name": "Yogyakarta (Tugu)", "country": "Indonesia", "lat": -7.7829, "lon": 110.3670},
        {"name": "Surabaya", "country": "Indonesia", "lat": -7.2575, "lon": 112.7521},
        {"name": "Bandung", "country": "Indonesia", "lat": -6.9175, "lon": 107.6191},
        {"name": "Semarang", "country": "Indonesia", "lat": -6.9667, "lon": 110.4167},
        {"name": "Medan", "country": "Indonesia", "lat": 3.5952, "lon": 98.6722},
        {"name": "Makassar", "country": "Indonesia", "lat": -5.1477, "lon": 119.4327},
        {"name": "Denpasar (Bali)", "country": "Indonesia", "lat": -8.6705, "lon": 115.2126},
        {"name": "IKN Nusantara", "country": "Indonesia", "lat": -0.9730, "lon": 116.7090},
        {"name": "Bone (Lokasi Bolide 2009)", "country": "Indonesia", "lat": -4.5386, "lon": 120.3279},
        {"name": "Prambanan (Lokasi Meteorit 1797)", "country": "Indonesia", "lat": -7.7520, "lon": 110.4915},
        {"name": "Karanganyar / Jumapalo", "country": "Indonesia", "lat": -7.6500, "lon": 111.0167},
        {"name": "Ngawi (Djati-Pengilon)", "country": "Indonesia", "lat": -7.4040, "lon": 111.4460},
        {"name": "Jepara", "country": "Indonesia", "lat": -6.5888, "lon": 110.6684},
        {"name": "Tokyo", "country": "Japan", "lat": 35.6762, "lon": 139.6503},
        {"name": "New York", "country": "USA", "lat": 40.7128, "lon": -74.0060},
        {"name": "London", "country": "UK", "lat": 51.5074, "lon": -0.1278},
        {"name": "Cairo", "country": "Egypt", "lat": 30.0444, "lon": 31.2357},
        {"name": "Chelyabinsk (Lokasi Bolide 2013)", "country": "Russia", "lat": 55.1644, "lon": 61.4368},
        {"name": "Tunguska (Lokasi Airburst 1908)", "country": "Russia", "lat": 60.8860, "lon": 101.8940},
    ]
    for city in all_target_cities:
      city["pop_density_km2"] = get_density(city["lat"], city["lon"])

    app_data["target_cities_sedac"] = all_target_cities
    app_data["sedac_latitude_profile_1deg"] = lat_profile_1deg

    output_lovable = REPO_ROOT / "meteorite_datamining_with_sedac.json"
    with open(output_lovable, "w", encoding="utf-8") as f:
      json.dump(app_data, f, indent=2)
    print(f"      -> Selesai! Tersimpan di: {output_lovable}")
  else:
    print(f"[4/4] Info: {lovable_json_path} tidak ditemukan. File gabungan tidak dibuat.")

[1/5] Membaca raster GeoTIFF: gpw_v4_population_density_rev10_2020_2pt5_min.tif ...
      Dimensi Raster: 8640x3480 piksel | Resolusi: 0.0417 derajat
[2/5] Info: Meteorite_Landings_Cleaned_Full.json tidak ada di folder, lanjut ke file aplikasi Lovable...
[3/5] Mengekstrak profil kepadatan penduduk seluruh pita garis lintang (-90° s/d 90°)...
[4/5] Mengekstrak matriks grid global (0.5° x 0.5°) untuk fitur klik bebas di peta...
      -> Tersimpan: sedac_global_grid_lookup.json (46704 grid daratan berpenghuni)
